# MATH 3260 Research Project — Starter Notebook 2
## Pattern Classification or Simple Prediction

This notebook begins after Starter Notebook 1 has processed your Google Trends data.

Starter Notebook 1 should have downloaded one file:

```text
MATH3260_Meme_Project_Processed_Data.zip
```

You do **not** need to unzip that file yourself. This notebook will ask you to upload it and will extract the processed data automatically.

The main file inside the ZIP is

```text
meme_vectors_36_month.csv
```

where each usable meme is represented by a vector

$$
\mathbf{x}_i\in\mathbb{R}^{36},
$$

and the complete dataset is

$$
X\in\mathbb{R}^{n\times36}.
$$

You will choose **one primary track**:

- **Track A — Pattern classification using $k$-means**: recommended; requires at least 100 usable memes.
- **Track B — Simple least-squares prediction**: optional; recommended with at least 120 usable memes.

Starter Notebook 1 may also include `meme_partial_recent_long.csv`. Recent incomplete curves are not used to train $k$-means, but Track A contains an optional comparison with the early portions of the learned centroids. No future values are invented.

> The Python code is supplied. Your main responsibility is to understand the Linear Algebra and interpret the results.


In [ ]:
from pathlib import Path
import zipfile
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

PROCESSED_FOLDER = Path("processed_meme_data")

DATA_FILE = PROCESSED_FOLDER / "meme_vectors_36_month.csv"
QUALITY_FILE = PROCESSED_FOLDER / "meme_quality_control.csv"
PARTIAL_FILE = PROCESSED_FOLDER / "meme_partial_recent_long.csv"

MIN_PARTIAL_MONTHS = 10

print("Expected processed-data folder:", PROCESSED_FOLDER)


## 1. Upload the processed-data ZIP from Notebook 1

Use the ZIP file that Starter Notebook 1 downloaded:

```text
MATH3260_Meme_Project_Processed_Data.zip
```

### Student steps

**Step 1.** Run the next code cell.

**Step 2.** Click **Choose Files**.

**Step 3.** Select the processed-data ZIP that you downloaded from Notebook 1.

**Step 4.** Wait for the upload and extraction to finish.

**Step 5.** Confirm that you see:

```text
PROCESSED DATA READY
Found: meme_vectors_36_month.csv
```

You do **not** need to manually upload individual CSV files, and you do **not** need to unzip the package yourself.

If you accidentally choose the wrong ZIP file, rerun the cell and select the correct one.


In [ ]:
# ============================================================
# STUDENT STEP: Upload the ONE processed-data ZIP from Notebook 1
# ============================================================

def extract_processed_data_zip(zip_path, target_folder=PROCESSED_FOLDER):
    """
    Extract the processed files created by Starter Notebook 1.
    Files may be stored directly in the ZIP or inside subfolders.
    """
    zip_path = Path(zip_path)
    target_folder = Path(target_folder)

    # Start fresh so old processed data are not mixed with a new upload.
    if target_folder.exists():
        shutil.rmtree(target_folder)

    target_folder.mkdir(parents=True, exist_ok=True)

    expected_names = {
        "meme_vectors_36_month.csv",
        "meme_quality_control.csv",
        "meme_partial_recent_long.csv",
    }

    extracted = []

    with zipfile.ZipFile(zip_path, "r") as archive:
        for member in archive.infolist():
            if member.is_dir():
                continue

            name = Path(member.filename).name

            if name not in expected_names:
                continue

            target = target_folder / name
            target.write_bytes(archive.read(member))
            extracted.append(target)

    return extracted


try:
    from google.colab import files
except ImportError:
    raise RuntimeError(
        "This upload cell is designed for Google Colab. "
        "Open Starter Notebook 2 in Colab and run this cell there."
    )

print("=" * 68)
print("UPLOAD THE PROCESSED-DATA ZIP FROM STARTER NOTEBOOK 1")
print("=" * 68)
print()
print("Click 'Choose Files' and select ONE processed-data ZIP file.")
print()

uploaded = files.upload()

zip_names = [
    name for name in uploaded.keys()
    if str(name).lower().endswith(".zip")
]

if len(zip_names) == 0:
    raise ValueError(
        "No ZIP file was uploaded. Rerun this cell and select "
        "'MATH3260_Meme_Project_Processed_Data.zip'."
    )

if len(zip_names) > 1:
    raise ValueError(
        "More than one ZIP file was uploaded. "
        "Rerun this cell and select only ONE processed-data ZIP."
    )

selected_zip = Path(zip_names[0])

if not zipfile.is_zipfile(selected_zip):
    raise ValueError(
        f"'{selected_zip.name}' is not a valid ZIP file."
    )

extracted = extract_processed_data_zip(
    selected_zip,
    PROCESSED_FOLDER
)

if not DATA_FILE.exists():
    raise ValueError(
        "The ZIP was uploaded, but 'meme_vectors_36_month.csv' "
        "was not found inside it. Make sure you selected the ZIP "
        "downloaded by Starter Notebook 1."
    )

print()
print("=" * 68)
print("PROCESSED DATA READY")
print("=" * 68)
print("Uploaded package:", selected_zip.name)
print("Found:", DATA_FILE.name)

if QUALITY_FILE.exists():
    print("Found:", QUALITY_FILE.name)

if PARTIAL_FILE.exists():
    print("Found:", PARTIAL_FILE.name)
else:
    print("No recent-partial file was included. That is okay.")

print()
print("Continue to the next cell to load the matrix X.")


## 2. Load the vector matrix

The columns named `t01` through `t36` are the components of the lifecycle vectors.

If there are $n$ usable memes, then

$$
X\in\mathbb{R}^{n\times36}.
$$

The next cell loads the matrix automatically from the ZIP package you just uploaded.

Before running any clustering or prediction algorithm, verify the dimensions and explain what they mean.


In [ ]:
if DATA_FILE.exists():
    data = pd.read_csv(DATA_FILE)
    vector_cols = [f"t{j:02d}" for j in range(1, 37)]

    missing_vector_cols = [
        col for col in vector_cols
        if col not in data.columns
    ]

    if missing_vector_cols:
        raise ValueError(
            "The vector file is missing expected lifecycle columns: "
            + ", ".join(missing_vector_cols)
        )

    X = data[vector_cols].to_numpy(dtype=float)

    print("=" * 60)
    print("VECTOR MATRIX LOADED SUCCESSFULLY")
    print("=" * 60)
    print("X shape:", X.shape)
    print("Number of meme vectors:", X.shape[0])
    print("Dimension of each vector:", X.shape[1])
    print()

    if X.shape[0] >= 100:
        print("Project minimum satisfied: at least 100 complete vectors.")
    else:
        print(
            "WARNING: fewer than 100 complete vectors were loaded. "
            "Return to Notebook 1 and add more candidate memes."
        )

    display(data.head())

else:
    raise FileNotFoundError(
        "The processed vector file was not found. "
        "Run the processed-data upload cell above first."
    )


# Part I — A Core Linear Algebra Tool: Euclidean Distance

Before $k$-means, we need a numerical way to compare two vectors.

For

$$
\mathbf{x},\mathbf{y}\in\mathbb{R}^{36},
$$

the Euclidean distance is

$$
d(\mathbf{x},\mathbf{y})
=
\|\mathbf{x}-\mathbf{y}\|_2
=
\sqrt{(x_1-y_1)^2+\cdots+(x_{36}-y_{36})^2}.
$$

This calculation has three Linear Algebra steps:

1. subtract the vectors;
2. compute the $L^2$ norm of the difference;
3. interpret the resulting nonnegative scalar as separation in $\mathbb{R}^{36}$.

### Small example

Suppose

$$
\mathbf{u}=
\begin{bmatrix}
10\\20\\30
\end{bmatrix}
$$

and

$$
\mathbf{v}=
\begin{bmatrix}
12\\18\\33
\end{bmatrix}.
$$

Then

$$
\mathbf{u}-\mathbf{v}
=
\begin{bmatrix}
-2\\2\\-3
\end{bmatrix},
$$

so

$$
\|\mathbf{u}-\mathbf{v}\|_2
=
\sqrt{(-2)^2+2^2+(-3)^2}
=
\sqrt{17}.
$$

The same operation is used for the 36-dimensional meme vectors.

In [ ]:
if "X" in globals() and len(X) >= 2:
    i = 0
    j = 1

    x = X[i]
    y = X[j]
    distance = np.linalg.norm(x - y)

    name_i = data.loc[i, "meme_name"] if "meme_name" in data.columns else f"row {i}"
    name_j = data.loc[j, "meme_name"] if "meme_name" in data.columns else f"row {j}"

    print("First meme:", name_i)
    print("Second meme:", name_j)
    print("Euclidean distance:", distance)

# Track A — Pattern Classification Using $k$-Means

This is the recommended project track.

For the course project, use

$$
k=4.
$$

The number four is motivated by the reference study, but the software labels 0, 1, 2, and 3 do not automatically correspond to the paper's named patterns. You must inspect the centroid curves and representative memes before interpreting a group.

## 2A. What is a centroid?

Suppose one cluster contains the vectors

$$
\mathbf{x}_1,\mathbf{x}_2,\ldots,\mathbf{x}_m.
$$

Its centroid is

$$
\boldsymbol{\mu}
=
\frac{1}{m}
(\mathbf{x}_1+\mathbf{x}_2+\cdots+\mathbf{x}_m).
$$

This is an ordinary vector operation:

1. add vectors component by component;
2. multiply by the scalar $1/m$.

Therefore

$$
\boldsymbol{\mu}\in\mathbb{R}^{36}.
$$

The $j$th component of the centroid is the average of the $j$th lifecycle component across all memes currently in the cluster.

So the centroid is naturally interpreted as an **average lifecycle curve**.

## 3A. The $k$-means algorithm — detailed explanation

$k$-means alternates between two simple ideas: **nearest-centroid assignment** and **centroid averaging**.

### Step 0: Initialize four centroids

The algorithm begins with four initial centroid vectors

$$
\boldsymbol{\mu}_1,
\boldsymbol{\mu}_2,
\boldsymbol{\mu}_3,
\boldsymbol{\mu}_4.
$$

Software chooses reasonable starting centroids. The exact initial choices may vary, which is why the code uses several initializations internally and keeps the best result.

### Step 1: Assignment

For every meme vector $\mathbf{x}_i$, compute four squared distances:

$$
\|\mathbf{x}_i-\boldsymbol{\mu}_1\|_2^2,
$$

$$
\|\mathbf{x}_i-\boldsymbol{\mu}_2\|_2^2,
$$

$$
\|\mathbf{x}_i-\boldsymbol{\mu}_3\|_2^2,
$$

$$
\|\mathbf{x}_i-\boldsymbol{\mu}_4\|_2^2.
$$

Assign $\mathbf{x}_i$ to the cluster whose centroid gives the smallest value.

Mathematically,

$$
\operatorname{cluster}(\mathbf{x}_i)
=
\operatorname*{arg\,min}_{1\leq j\leq4}
\|\mathbf{x}_i-\boldsymbol{\mu}_j\|_2^2.
$$

The notation $\operatorname{arg\,min}$ means: choose the **index** $j$ that produces the smallest distance.

### Step 2: Update

After all vectors have been assigned, recompute each centroid as the average of the vectors in that group.

For cluster $C_j$ containing $m_j$ vectors,

$$
\boldsymbol{\mu}_j
=
\frac{1}{m_j}
\sum_{\mathbf{x}_i\in C_j}\mathbf{x}_i.
$$

### Step 3: Repeat

The assignment step may change because the centroids moved. Therefore, repeat:

$$
\boxed{
\text{assign to nearest centroid}
\longrightarrow
\text{recompute centroids}
}
$$

until the groups stop changing substantially.

### What is the algorithm trying to minimize?

$k$-means tries to make vectors within each group close to their own centroid. Its objective is

$$
J
=
\sum_{i=1}^{n}
\min_{1\leq j\leq4}
\|\mathbf{x}_i-\boldsymbol{\mu}_j\|_2^2.
$$

You do **not** need to derive an optimization theorem for this project. You should understand that smaller within-cluster squared distances mean more compact groups.

## 4A. A tiny two-dimensional illustration

Before thinking about 36 dimensions, imagine four points in $\mathbb{R}^2$:

$$
\mathbf{x}_1=(1,1),
$$

$$
\mathbf{x}_2=(2,1),
$$

$$
\mathbf{x}_3=(8,8),
$$

$$
\mathbf{x}_4=(9,8).
$$

If $k=2$, a natural grouping is

$$
C_1=\{\mathbf{x}_1,\mathbf{x}_2\},
$$

$$
C_2=\{\mathbf{x}_3,\mathbf{x}_4\}.
$$

The first centroid is

$$
\boldsymbol{\mu}_1
=
\frac{1}{2}
\left[
\begin{array}{c}
1\\1
\end{array}
+
\begin{array}{c}
2\\1
\end{array}
\right]
=
\begin{bmatrix}
1.5\\1
\end{bmatrix}.
$$

The second centroid is

$$
\boldsymbol{\mu}_2
=
\begin{bmatrix}
8.5\\8
\end{bmatrix}.
$$

Every meme project vector follows the same idea. The only difference is that the vectors live in $\mathbb{R}^{36}$ instead of $\mathbb{R}^{2}$.

In [ ]:
if "X" in globals() and len(X) >= 4:
    K = 4

    kmeans = KMeans(
        n_clusters=K,
        random_state=3260,
        n_init=20,
    )

    labels = kmeans.fit_predict(X)
    centroids = kmeans.cluster_centers_

    print("Centroid matrix shape:", centroids.shape)
    print("The four centroid vectors are the four rows of this matrix.")

## 5A. Interpret the centroid matrix

Because there are four clusters and each centroid lies in $\mathbb{R}^{36}$,

$$
M=
\begin{bmatrix}
---\boldsymbol{\mu}_1^T---\\
---\boldsymbol{\mu}_2^T---\\
---\boldsymbol{\mu}_3^T---\\
---\boldsymbol{\mu}_4^T---
\end{bmatrix}
\in\mathbb{R}^{4\times36}.
$$

This is another matrix interpretation:

- each row is one average lifecycle pattern;
- each column compares the four cluster averages at the same relative lifecycle month.

In [ ]:
if "centroids" in globals():
    plt.figure(figsize=(10, 5))

    months = np.arange(1, 37)
    for c in range(K):
        plt.plot(months, centroids[c], label=f"Cluster {c}")

    plt.xlabel("Month within the aligned 36-month lifecycle window")
    plt.ylabel("Peak-rescaled Google Trends interest")
    plt.title("Four k-means Centroid Curves")
    plt.legend()
    plt.show()

## 6A. Cluster sizes

The number of vectors assigned to each cluster tells us how common each discovered pattern is within **your selected dataset**.

Do not interpret the percentages as estimates for all internet memes. Your meme list is a student-selected sample, not a random sample of the entire internet.

In [ ]:
if "labels" in globals():
    cluster_counts = pd.Series(labels).value_counts().sort_index()
    cluster_table = pd.DataFrame({
        "cluster": cluster_counts.index,
        "number_of_memes": cluster_counts.values,
        "percentage": 100 * cluster_counts.values / len(labels),
    })
    display(cluster_table)

## 7A. Representative meme = vector nearest to the centroid

A useful representative of cluster $j$ is the meme vector with the smallest distance to

$$
\boldsymbol{\mu}_j.
$$

That is,

$$
\mathbf{x}_{\mathrm{rep}}
=
\operatorname*{arg\,min}_{\mathbf{x}_i\in C_j}
\|\mathbf{x}_i-\boldsymbol{\mu}_j\|_2.
$$

This is a direct application of the same Euclidean-distance idea.

In [ ]:
if "labels" in globals():
    representatives = []

    for c in range(K):
        indices = np.where(labels == c)[0]
        members = X[indices]
        distances = np.linalg.norm(members - centroids[c], axis=1)
        best_local = int(np.argmin(distances))
        best_global = int(indices[best_local])

        meme_name = data.loc[best_global, "meme_name"] if "meme_name" in data.columns else f"row {best_global}"

        representatives.append({
            "cluster": c,
            "representative_meme": meme_name,
            "distance_to_centroid": distances[best_local],
        })

    representative_table = pd.DataFrame(representatives)
    display(representative_table)

## 8A. Verify one assignment manually

Choose one meme vector $\mathbf{x}$ and compute its four distances:

$$
d_1=\|\mathbf{x}-\boldsymbol{\mu}_1\|_2,
$$

$$
d_2=\|\mathbf{x}-\boldsymbol{\mu}_2\|_2,
$$

$$
d_3=\|\mathbf{x}-\boldsymbol{\mu}_3\|_2,
$$

$$
d_4=\|\mathbf{x}-\boldsymbol{\mu}_4\|_2.
$$

The smallest value determines the nearest centroid.

This check is important because it connects the software label back to a calculation you understand from Linear Algebra.

In [ ]:
if "labels" in globals():
    sample_index = 0
    x = X[sample_index]
    distances = np.linalg.norm(centroids - x, axis=1)

    name = data.loc[sample_index, "meme_name"] if "meme_name" in data.columns else f"row {sample_index}"
    print("Selected meme:", name)

    distance_table = pd.DataFrame({
        "cluster": np.arange(K),
        "distance_to_centroid": distances,
    })
    display(distance_table)

    print("Nearest centroid:", int(np.argmin(distances)))
    print("k-means assigned label:", int(labels[sample_index]))

## 9A. Average distance to centroid — a simple measure of cluster spread

For cluster $C_j$, compute

$$
\frac{1}{m_j}
\sum_{\mathbf{x}_i\in C_j}
\|\mathbf{x}_i-\boldsymbol{\mu}_j\|_2.
$$

A smaller average distance means the vectors in that cluster are more tightly concentrated around their average lifecycle curve.

A larger value means the cluster contains more variation.

This measure is intentionally simple and uses only vector subtraction, norms, and averages.

In [ ]:
if "labels" in globals():
    spread_records = []

    for c in range(K):
        members = X[labels == c]
        distances = np.linalg.norm(members - centroids[c], axis=1)

        spread_records.append({
            "cluster": c,
            "number_of_memes": len(members),
            "average_distance_to_centroid": distances.mean(),
            "maximum_distance_to_centroid": distances.max(),
        })

    spread_table = pd.DataFrame(spread_records)
    display(spread_table)

## 10A. What you must interpret for Track A

Your report should not stop with four software labels.

For each cluster:

1. inspect its centroid curve;
2. inspect at least one representative meme;
3. describe the temporal shape in ordinary language;
4. compare its average distance to centroid with the other groups;
5. decide whether the group resembles one of the broad patterns discussed in the reference study or whether your data suggest a different interpretation.

Remember: cluster labels are arbitrary. The mathematical result is the grouping and the centroid geometry, not the number attached to a cluster.

# Optional Track A Extension — Classify a Recent Incomplete Meme Without Inventing Its Future

A recent meme may have only $m<36$ observed lifecycle months.

It should **not** be placed into the original $k$-means training matrix because $k$-means expects all samples to have the same dimension.

However, once the four complete-data centroids have been learned, we may compare the observed partial vector

$$
\mathbf{z}\in\mathbb{R}^{m}
$$

with the first $m$ components of each centroid:

$$
\boldsymbol{\mu}_{1,1:m},
\boldsymbol{\mu}_{2,1:m},
\boldsymbol{\mu}_{3,1:m},
\boldsymbol{\mu}_{4,1:m}.
$$

For cluster $j$, compute

$$
d_j^{(m)}
=
\left\|
\mathbf{z}-\boldsymbol{\mu}_{j,1:m}
\right\|_2.
$$

The smallest distance identifies the historical pattern that the recent meme most resembles **so far**.

### Why divide by $\sqrt{m}$ in the summary table?

For one fixed recent meme, dividing all four distances by $\sqrt{m}$ does not change which centroid is nearest. But

$$
\frac{1}{\sqrt{m}}
\left\|
\mathbf{z}-\boldsymbol{\mu}_{j,1:m}
\right\|_2
$$

is a per-component root-mean-square distance. It makes distances from partial curves of different observed lengths somewhat easier to compare.

### Important interpretation

This is an **early-pattern resemblance**, not a forecast of the exact missing months. The future centroid values are not claimed to be the meme's future values.

In [ ]:
def classify_partial_curve(partial_values, centroids):
    partial_values = np.asarray(partial_values, dtype=float)
    m = len(partial_values)

    if m == 0:
        raise ValueError("The partial curve contains no observed values.")
    if m > centroids.shape[1]:
        partial_values = partial_values[:centroids.shape[1]]
        m = len(partial_values)

    truncated_centroids = centroids[:, :m]
    distances = np.linalg.norm(truncated_centroids - partial_values, axis=1)
    rms_distances = distances / np.sqrt(m)

    return distances, rms_distances, int(np.argmin(distances))


## Recent-meme batch comparison

Notebook 1 saves recent incomplete observations in long format:

| meme | lifecycle month | normalized interest |
|---|---:|---:|
| Recent Meme | 1 | ... |
| Recent Meme | 2 | ... |
| Recent Meme | 3 | ... |

The code below groups rows belonging to the same meme, creates its partial vector, and compares that vector with the first $m$ components of every centroid.

We require at least 10 observed lifecycle months for this optional comparison. This threshold is a course-level quality rule, not a mathematical theorem.

In [ ]:
if PARTIAL_FILE.exists() and "centroids" in globals():
    partial_data = pd.read_csv(PARTIAL_FILE)
    recent_results = []

    group_keys = [c for c in ["file_name", "meme_name", "search_term"] if c in partial_data.columns]

    for keys, group in partial_data.groupby(group_keys, dropna=False):
        group = group.sort_values("lifecycle_month")
        value_column = (
            "peak_rescaled_interest"
            if "peak_rescaled_interest" in group.columns
            else "normalized_interest"
        )
        values = group[value_column].to_numpy(dtype=float)
        m = min(len(values), 36)
        values = values[:m]

        if m < MIN_PARTIAL_MONTHS:
            continue

        distances, rms_distances, nearest = classify_partial_curve(values, centroids)

        if not isinstance(keys, tuple):
            keys = (keys,)
        key_dict = dict(zip(group_keys, keys))

        row = {
            **key_dict,
            "observed_lifecycle_months": m,
            "nearest_historical_cluster_so_far": nearest,
        }
        for c in range(K):
            row[f"rms_distance_cluster_{c}"] = rms_distances[c]
        recent_results.append(row)

    recent_classification = pd.DataFrame(recent_results)
    if len(recent_classification):
        display(recent_classification)
        recent_classification.to_csv("recent_partial_cluster_resemblance.csv", index=False)
        print("Saved recent_partial_cluster_resemblance.csv")
    else:
        print("No partial recent curve has at least", MIN_PARTIAL_MONTHS, "observed lifecycle months.")
else:
    print("This optional section runs after k-means and when meme_partial_recent_long.csv is available.")


## Plot one recent partial curve against the centroid references

The solid partial curve below contains only observed values. The centroid curves are historical reference patterns.

Do not describe the unobserved part of a centroid as the recent meme's predicted future. A careful interpretation is:

> During its first $m$ observed lifecycle months, this meme is currently closest to Cluster $j$.

Its eventual 36-month trajectory may still change.

In [ ]:
if PARTIAL_FILE.exists() and "centroids" in globals():
    partial_data = pd.read_csv(PARTIAL_FILE)
    if len(partial_data):
        first_file = partial_data["file_name"].iloc[0] if "file_name" in partial_data.columns else None
        if first_file is not None:
            g = partial_data[partial_data["file_name"] == first_file].sort_values("lifecycle_month")
            value_column = (
                "peak_rescaled_interest"
                if "peak_rescaled_interest" in g.columns
                else "normalized_interest"
            )
            values = g[value_column].to_numpy(dtype=float)[:36]
            m = len(values)

            if m >= MIN_PARTIAL_MONTHS:
                _, _, nearest = classify_partial_curve(values, centroids)
                months_full = np.arange(1, 37)
                months_partial = np.arange(1, m + 1)

                plt.figure(figsize=(10, 5))
                for c in range(K):
                    plt.plot(months_full, centroids[c], alpha=0.55, label=f"Historical centroid {c}")
                plt.plot(months_partial, values, linewidth=3, label="Observed recent meme")
                plt.axvline(m, linestyle="--", label="Last observed lifecycle month")
                plt.xlabel("Lifecycle month")
                plt.ylabel("Peak-rescaled trend")
                plt.title(f"Recent Meme vs. Historical Centroids — closest so far: Cluster {nearest}")
                plt.legend()
                plt.show()


# Track B — Optional Simple Least-Squares Prediction

Choose this track only if you have at least 120 usable lifecycle vectors and prefer a prediction question.

We keep the prediction deliberately simple so that the Linear Algebra remains visible.

### Course-level prediction design

Use the first 8 lifecycle components

$$
x_1,x_2,\ldots,x_8
$$

as predictors.

Let the response be the average of the last four components:

$$
y
=
\frac{x_{33}+x_{34}+x_{35}+x_{36}}{4}.
$$

Thus the question is approximately:

> Can the early part of the aligned meme lifecycle help predict its later popularity level?

## 2B. Construct the design matrix

For meme $i$, the predictor row is

$$
[1,x_{i1},x_{i2},\ldots,x_{i8}].
$$

The leading 1 allows the model to include an intercept.

With $n$ memes,

$$
A\in\mathbb{R}^{n\times9}.
$$

The response vector is

$$
\mathbf{y}
=
\begin{bmatrix}
y_1\\
y_2\\
\vdots\\
y_n
\end{bmatrix}
\in\mathbb{R}^{n}.
$$

We seek coefficients

$$
\boldsymbol{\beta}\in\mathbb{R}^{9}
$$

so that

$$
A\boldsymbol{\beta}
$$

is close to $\mathbf{y}$.

## 3B. Why least squares?

Usually there is no coefficient vector satisfying

$$
A\boldsymbol{\beta}=\mathbf{y}
$$

exactly for every meme.

Instead, minimize the residual norm:

$$
\min_{\boldsymbol{\beta}}
\|A\boldsymbol{\beta}-\mathbf{y}\|_2^2.
$$

The vector

$$
\widehat{\mathbf{y}}
=
A\widehat{\boldsymbol{\beta}}
$$

is the fitted response.

Geometrically,

$$
\widehat{\mathbf{y}}
$$

is the point in the column space of $A$ closest to $\mathbf{y}$.

Therefore the residual

$$
\mathbf{r}
=
\mathbf{y}-\widehat{\mathbf{y}}
$$

must be perpendicular to every column of $A$:

$$
A^T\mathbf{r}=\mathbf{0}.
$$

Substituting

$$
\mathbf{r}=\mathbf{y}-A\widehat{\boldsymbol{\beta}}
$$

gives

$$
A^T(\mathbf{y}-A\widehat{\boldsymbol{\beta}})=\mathbf{0},
$$

and therefore

$$
\boxed{
A^TA\widehat{\boldsymbol{\beta}}=A^T\mathbf{y}.
}
$$

These are the normal equations.

The notebook uses a numerically stable least-squares solver rather than explicitly computing $(A^TA)^{-1}$. The Linear Algebra interpretation is still exactly the projection picture above.

In [ ]:
if "X" in globals() and len(X) >= 120:
    P = 8
    Q = 4

    early = X[:, :P]
    y = X[:, -Q:].mean(axis=1)
    A = np.column_stack([np.ones(len(X)), early])

    print("Design matrix A shape:", A.shape)
    print("Response vector y shape:", y.shape)
else:
    print("Track B is recommended only with at least 120 usable meme vectors.")

## 4B. Training data and testing data

If the same observations are used both to fit and evaluate a prediction rule, the reported error can be overly optimistic.

Therefore, the starter notebook uses a simple split:

- 70% of the meme vectors for fitting the least-squares coefficients;
- 30% for evaluating predictions on observations not used in the fit.

This is not an advanced machine-learning validation procedure. It is simply a fairer way to evaluate a prediction rule.

In [ ]:
if "A" in globals():
    A_train, A_test, y_train, y_test = train_test_split(
        A,
        y,
        test_size=0.30,
        random_state=3260,
    )

    print("Training rows:", A_train.shape[0])
    print("Testing rows:", A_test.shape[0])

In [ ]:
if "A_train" in globals():
    beta, residuals, rank, singular_values = np.linalg.lstsq(
        A_train,
        y_train,
        rcond=None,
    )

    print("Number of fitted coefficients:", len(beta))
    print("Rank of the training design matrix:", rank)

## 5B. Verify the residual orthogonality

For the training set,

$$
\mathbf{r}_{\mathrm{train}}
=
\mathbf{y}_{\mathrm{train}}
-
A_{\mathrm{train}}\widehat{\boldsymbol{\beta}}.
$$

Least squares predicts that

$$
A_{\mathrm{train}}^T\mathbf{r}_{\mathrm{train}}
\approx
\mathbf{0}.
$$

Because a computer uses finite-precision arithmetic, the result will usually be a very small vector rather than exact zeros.

Checking this quantity connects the numerical calculation directly to the orthogonality theorem from class.

In [ ]:
if "beta" in globals():
    yhat_train = A_train @ beta
    r_train = y_train - yhat_train

    check = A_train.T @ r_train

    print("Norm of A_train^T r_train:", np.linalg.norm(check))
    print("A very small value supports the expected residual orthogonality.")

## 6B. Evaluate prediction error

For the test data,

$$
\widehat{\mathbf{y}}_{\mathrm{test}}
=
A_{\mathrm{test}}\widehat{\boldsymbol{\beta}}.
$$

Two simple error measures are used.

### Mean absolute error

$$
\operatorname{MAE}
=
\frac{1}{m}
\sum_{i=1}^{m}
|y_i-\widehat{y}_i|.
$$

### Root mean squared error

$$
\operatorname{RMSE}
=
\sqrt{
\frac{1}{m}
\sum_{i=1}^{m}
(y_i-\widehat{y}_i)^2
}.
$$

Both measure prediction error in the same normalized popularity units as the response. Smaller values indicate better agreement between predicted and observed later popularity.

In [ ]:
if "beta" in globals():
    yhat_test = A_test @ beta

    mae = mean_absolute_error(y_test, yhat_test)
    rmse = np.sqrt(mean_squared_error(y_test, yhat_test))

    print("Test MAE:", mae)
    print("Test RMSE:", rmse)

    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, yhat_test)
    low = min(y_test.min(), yhat_test.min())
    high = max(y_test.max(), yhat_test.max())
    plt.plot([low, high], [low, high], linestyle="--")
    plt.xlabel("Observed later popularity")
    plt.ylabel("Predicted later popularity")
    plt.title("Observed vs. Predicted Later Popularity")
    plt.show()

# Final Required Reflection

Regardless of your chosen track, answer these questions in your report or notebook.

1. Why does each meme belong to $\mathbb{R}^{36}$ after preprocessing?
2. What do the rows and columns of $X$ represent?
3. Why is Euclidean distance meaningful only after the curves have comparable dimensions and scaling?
4. For Track A, why is a centroid a linear combination of vectors?
5. For Track A, what two operations repeat during $k$-means?
6. For Track B, why is least squares a projection problem?
7. What mathematical information is **not** contained in the time-series vector? For example, does the vector know the meme's image, meaning, or cultural context?
8. What is one limitation of the simplified algorithm used in this undergraduate project?

The goal is to demonstrate that the algorithms are understandable combinations of the Linear Algebra ideas studied in MATH 3260.